### DLT Pipeline

In [0]:
import dlt
from pyspark.sql.functions import *

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-6808539384278457>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import *

ModuleNotFoundError: No module named 'dlt'

### Streaming Table 

In [0]:
# Expectations 
rules = {
    "valid_id": "user_id IS NOT NULL",
    "valid_sequence": "created_at IS NOT NULL",
}

In [0]:
@dlt.table(name="DimUser_stage")

@dlt.expect_all_or_drop(rules)

# This function will do above this source
def DimUser_stage():
    df = spark.readStream.table("travel_journal_catalog.silver.accounts")
    return df




In [0]:
spark.table("travel_journal_catalog.silver.images").printSchema()


## Streaming View

In [0]:
import dlt
import pyspark.sql.functions as F
from pyspark.sql.window import Window


@dlt.view(name="DimUser_stage_view")
def DimUser_stage_view():



    latest_bio_df = dlt.read("travel_journal_catalog.silver.bio").dropDuplicates(["user_id"]).select("user_id","bio_text")
    

    image_df = dlt.read("travel_journal_catalog.silver.images").select("user_id","image_url")
    
    stage_df = dlt.read_stream("DimUser_stage")

    df = (
            stage_df
            .join(latest_bio_df, on="user_id", how="left")
            .join(image_df, on="user_id", how="left")
            .withColumn(
                "DimUserKey",
                F.sha2(
                    F.concat_ws("||",
                        F.col("user_id").cast("string"),
                        F.col("created_at").cast("string")
                    ),          # <-- concat_ws CLOSES here
                    256          # <-- numBits: 2nd arg of sha2
                )               # <-- sha2 closes
            )                   # <-- withColumn closes  (this ')' was missing)
            .drop("passwordhash", "email", "preferences", "image_id", "_rescued_data")
        )
    df = df.select("user_id","created_at","image_url","bio_text","DimUserKey","last_login","role","username","date_type")
    return df

In [0]:
dlt.create_streaming_table("dim_user")

dlt.apply_changes(
    target = "dim_user",
    source = "DimUser_stage_view",
    keys = ["user_id"],
    sequence_by = "created_at",
    stored_as_scd_type="2"
)